In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import StringType

In [0]:
%run ../functions/functions

In [0]:

database_name = "fato"
table_name = "ft_empresas"
target_path = f"{database_name}.{table_name}"
pk = " "

In [0]:
container_destino = "gold"
container_origem = "silver"

caminho_origem = f"abfss://{container_origem}@{STORAGE}.dfs.core.windows.net/cnpj"

In [0]:
df_estabelecimentos = spark.read.format("delta").load(f"{caminho_origem}/ESTABELECIMENTOS_CONSOLIDADA")
df_empresas = spark.read.format("delta").load(f"{caminho_origem}/EMPRESAS_CONSOLIDADA")
df_municipios = spark.read.format("delta").load(f"{caminho_origem}/MUNICIPIOS_CONSOLIDADA")
df_simples = spark.read.format("delta").load(f"{caminho_origem}/SIMPLES_CONSOLIDADA")


df_estabelecimentos = df_estabelecimentos.withColumn("municipio", f.expr("try_cast(municipio as int)"))

df_join2 = df_estabelecimentos.join(
    df_municipios, 
    df_estabelecimentos.municipio == df_municipios.codigo_municipio, 
    "inner"
)

df_join1 = df_join2.join(df_empresas, "cnpj_basico", "inner")


df_simples_tratado = df_simples.select(
    f.col("cnpj_basico"),
    f.when(f.col("opcao_simples") == "S", 1).otherwise(0).alias("ind_simples"),
    f.when(f.col("opcao_mei") == "S", 1).otherwise(0).alias("ind_mei"),
)

df_com_simples = df_join1.join(df_simples_tratado, "cnpj_basico","left")

df_final_com_flags = df_com_simples \
    .withColumn("fl_simples", f.coalesce(f.col("ind_simples"), f.lit(0))) \
    .withColumn("fl_mei", f.coalesce(f.col("ind_mei"), f.lit(0))) \
    .drop("ind_simples", "ind_mei")







df_resultado_final = df_final_com_flags.select(
    "cnpj_basico",
    "razao_social",
    "natureza_juridica",
    "qualificacao_responsavel",
    "capital_social",
    "porte_empresa",
    "cnpj_ordem",
    "cnpj_dv",
    "identificador_matriz_filial",
    "nome_fantasia",
    "situacao_cadastral",
    "data_situacao_cadastral",
    "motivo_situacao_cadastral",
    "nome_cidade_exterior",
    "pais",
    "data_inicio_atividade",
    "cnae_fiscal_principal",
    "cnae_fiscal_secundaria",
    "tipo_logradouro",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "uf",
    "municipio",
    "descricao_municipio",
    "ddd_1",
    "telefone_1",
    "ddd_2",
    "telefone_2",
    "correio_eletronico",
    "situacao_especial",
    "data_situacao_especial",
    "fl_simples",
    "fl_mei"     
).distinct()

df_resultado_final.display()

In [0]:

if not spark.catalog.tableExists(target_path):
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_path)
else:
    target_table = DeltaTable.forName(spark, target_path)
    target_table.alias("target").merge(
        df_final.alias("source"),
        f"target.{pk} = source.{pk}"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()